# VQA Counting Results 
### imports

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from utils.load_env_vars import load_env

load_env()

from torch.utils.data import DataLoader
import json 
import re

from tqdm import tqdm

from judge.query_sglang import query_judge_batched 
from judge.metrics import compute_count_metrics, compute_bool_metrics
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# visualize table for scene understanding

### data loading


In [ ]:
# Model name to LaTeX command mapping
MODEL_LATEX_MAPPING = {
    'OpenGVLab_InternVideo2_5_Chat_8B': r'\ivideo',
    'OpenGVLab_InternVL3-8B': r'\ivlsmall',
    'OpenGVLab_InternVL3-78B': r'\ivlbig',
    'lmms-lab_LLaVA-Video-7B-Qwen2': r'\llavavidsmall',
    'lmms-lab_LLaVA-Video-72B-Qwen2': r'\llavavidbig',
    'llava-hf_llava-onevision-qwen2-7b-ov-chat-hf': r'\llavaovsmall',
    'llava-hf_llava-onevision-qwen2-72b-ov-chat-hf': r'\llavaovbig',
    'gemini-2.0-flash': r'\geminiold',
    'gemini-2.5-flash': r'\gemininew',
}

In [ ]:
import json
import pandas as pd
from pathlib import Path

def load_all_metrics(verbose=True):
    """
    Load all metric JSON files and create a simple table per dataset
    
    Args:
        verbose: If True, prints loading progress and results. If False, loads silently.
    """
    base_path = "output/eval/scene_understanding"
    
    models_strings = ['OpenGVLab_InternVideo2_5_Chat_8B',
                      "OpenGVLab_InternVL3-8B",
                      "OpenGVLab_InternVL3-78B",
                      "lmms-lab_LLaVA-Video-7B-Qwen2",
                      "lmms-lab_LLaVA-Video-72B-Qwen2",
                      "llava-hf_llava-onevision-qwen2-7b-ov-chat-hf",
                      "llava-hf_llava-onevision-qwen2-72b-ov-chat-hf",
                      'gemini-2.0-flash',
                    #   'gemini-2.0-flash_advantage',
                      'gemini-2.5-flash',
                    #   'gemini-2.5-flash_advantage'
                      ]
    
    sample_frames = [32]
    datasets = ["unicycle", "bicycle", "tricycle"] #
    splits = ["test"]
    
    dataset_results = {}
    
    for dataset in datasets:
        if verbose:
            print(f"\nLoading metrics for dataset: {dataset}")
        dataset_metrics = []
        
        for sampled_frame in sample_frames:
            for split in splits:
                for model in models_strings:
                    
                    
                    if model == "lmms-lab_LLaVA-Video-7B-Qwen2" or model =="lmms-lab_LLaVA-Video-72B-Qwen2": # this models needs 16 frames
                        sf = 16
                    else: 
                        sf = sampled_frame

                    
                    
                    metric_file = f"scene_understanding_{dataset}_{sf}sf_metrics.json"
                    metrics_path = Path(base_path, dataset, split, model, metric_file)
                    
                    # Also check for answer CSV file
                    answer_csv_file = f"scene_understanding_{dataset}_{sf}sf.csv"
                    answer_csv_path = Path(base_path, dataset, split, model, answer_csv_file)
                    
                    if metrics_path.exists():
                        try:
                            with open(metrics_path, 'r') as f:
                                metrics = json.load(f)
                            
                            # Check if answer CSV exists and count lines
                            csv_exists = answer_csv_path.exists()
                            csv_line_count = 0
                            if csv_exists:
                                try:
                                    with open(answer_csv_path, 'r') as csv_f:
                                        csv_line_count = sum(1 for _ in csv_f) - 1  # Subtract 1 for header
                                except Exception as e:
                                    if verbose:
                                        print(f"  Warning: Could not count lines in {answer_csv_path}: {e}")
                            
                            # Process metrics: convert to percentage with 1 decimal
                            processed_metrics = {}
                            for key, value in metrics.items():
                                if isinstance(value, (float, int)) and not isinstance(value, bool):
                                    # Convert precision, recall, f1_score fields to percentage
                                    if 'precision' in key or 'recall' in key or 'f1_score' in key:
                                        processed_metrics[key] = round(float(value) * 100, 1)
                                    else:
                                        # Other numeric values (hits, evaluated, etc.)
                                        processed_metrics[key] = round(float(value), 1)
                                else:
                                    processed_metrics[key] = value
                            
                            row = {
                                'model': model,
                                'sample_frames': sf,
                                'csv_exists': f'✓ ({csv_line_count})' if csv_exists else '✗',
                                **processed_metrics
                            }
                            dataset_metrics.append(row)
                            if verbose:
                                csv_status = f'✓ ({csv_line_count} lines)' if csv_exists else '✗ CSV missing'
                                print(f"✓ Loaded: {model} | CSV: {csv_status}")
                            
                        except Exception as e:
                            if verbose:
                                print(f"✗ Error loading {metrics_path}: {e}")
                    else:
                        if verbose:
                            csv_status = '✓' if answer_csv_path.exists() else '✗'
                            print(f"✗ Metrics not found: {model} | CSV: {csv_status}")
        
        if dataset_metrics:
            df = pd.DataFrame(dataset_metrics)
            numeric_columns = df.select_dtypes(include=['float64', 'int64']).columns
            df[numeric_columns] = df[numeric_columns].round(1)
            dataset_results[dataset] = df
            
            # print(f"\nResults for {dataset.upper()}:")
            # print("-" * 50)
            # print(df.to_string(index=False))
            
        else:
            if verbose:
                print(f"No metrics files found for dataset: {dataset}")

    return dataset_results


results = load_all_metrics()

# Print summary
if results:
    print(f"\nSUMMARY:")
    for dataset, df in results.items():
        csv_present = df['csv_exists'].str.startswith('✓').sum()
        csv_missing = df['csv_exists'].str.startswith('✗').sum()
        print(f"{dataset.upper()}: {len(df)} metric records loaded | CSV files: {csv_present} present, {csv_missing} missing")
else:
    print("No results loaded!")

#pred output/eval/scene_understanding/unicycle/test/OpenGVLab_InternVideo2_5_Chat_8B/scene_understanding_unicycle_32sf_metrics.json
#true output/eval/scene_understanding/unicycle/test/OpenGVLab_InternVideo2_5_Chat_8B/scene_undetstanding_unicycle_32sf_metrics.json


### On-Demand Loading Examples

You can now load specific templates on demand:

```python
# Load only cycles template
results_cycles = load_all_metrics()

```

### plotting the table

In [ ]:
def create_combined_accuracy_table(results):
    """
    Create combined tables for precision, recall, and F1 score (without printing individual tables)
    """
    if not results:
        print("No results to create tables from!")
        return None, None, None, None, None, None
    
    # Collect data for each metric by dataset
    precision_data = {}
    recall_data = {}
    f1_data = {}
    
    for dataset_name, df in results.items():
        # Check if required columns exist
        required_cols = ['obj_precision', 'obj_recall', 'obj_f1_score']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f"Missing columns for {dataset_name}: {missing_cols}")
            continue
        
        # Clean model names
        df_clean = df.copy()
        df_clean['model_clean'] = df_clean['model'].apply(lambda x: x.split('/')[-1])
        
        # Group by model and get mean values (in case there are multiple entries per model)
        for model in df_clean['model_clean'].unique():
            model_data = df_clean[df_clean['model_clean'] == model]
            
            if model not in precision_data:
                precision_data[model] = {}
                recall_data[model] = {}
                f1_data[model] = {}
            
            # Values are already converted to percentage in load_all_metrics, so just take mean
            precision_data[model][dataset_name] = model_data['obj_precision'].mean()
            recall_data[model][dataset_name] = model_data['obj_recall'].mean()
            f1_data[model][dataset_name] = model_data['obj_f1_score'].mean()
    
    # Create DataFrames from the collected data
    combined_precision = pd.DataFrame(precision_data).T.round(1) if precision_data else None
    combined_recall = pd.DataFrame(recall_data).T.round(1) if recall_data else None
    combined_f1 = pd.DataFrame(f1_data).T.round(1) if f1_data else None
    
    return combined_precision, None, combined_recall, None, combined_f1, None


In [ ]:

results = load_all_metrics(verbose=False)

# Create combined tables for precision, recall, and F1 score
combined_precision, latex_precision, combined_recall, latex_recall, combined_f1, latex_f1 = create_combined_accuracy_table(results)


### Table with Precision, Recall, and F1 Score Side by Side


In [ ]:
def create_combined_metrics_table(combined_precision, combined_recall, combined_f1):
    """
    Create a table with models as rows and columns for precision, recall, and F1 per dataset
    """
    if combined_precision is None or combined_recall is None or combined_f1 is None:
        print("No data to create table!")
        return None
    
    print("\nCOMBINED METRICS TABLE - Precision, Recall & F1 Score")
    print("=" * 120)
    
    # Get datasets from column names
    datasets = []
    for col in combined_precision.columns:
        dataset = col.split('_')[0]
        if dataset not in datasets:
            datasets.append(dataset)
    
    # Get models
    models = combined_precision.index.tolist()
    
    # Create rows: one per model with 3 columns per dataset
    table_rows = []
    
    for model in models:
        row = {'Model': model}
        
        for dataset in datasets:
            # Get columns for this dataset
            prec_cols = [col for col in combined_precision.columns if col.startswith(dataset)]
            rec_cols = [col for col in combined_recall.columns if col.startswith(dataset)]
            f1_cols = [col for col in combined_f1.columns if col.startswith(dataset)]
            
            if prec_cols:
                prec_val = combined_precision.loc[model, prec_cols[0]]
                row[f"{dataset.upper()}\nPrecision (%)"] = f"{prec_val:.1f}" if prec_val > 0 else "--"
            else:
                row[f"{dataset.upper()}\nPrecision (%)"] = "--"
            
            if rec_cols:
                rec_val = combined_recall.loc[model, rec_cols[0]]
                row[f"{dataset.upper()}\nRecall (%)"] = f"{rec_val:.1f}" if rec_val > 0 else "--"
            else:
                row[f"{dataset.upper()}\nRecall (%)"] = "--"
            
            if f1_cols:
                f1_val = combined_f1.loc[model, f1_cols[0]]
                row[f"{dataset.upper()}\nF1 Score (%)"] = f"{f1_val:.1f}" if f1_val > 0 else "--"
            else:
                row[f"{dataset.upper()}\nF1 Score (%)"] = "--"
        
        table_rows.append(row)
    
    # Create DataFrame
    combined_df = pd.DataFrame(table_rows)
    
    # Print the table
    print("\nTable Format: Models (rows) × Dataset Metrics (columns)")
    print("-" * 120)
    print(combined_df.to_string(index=False))
    
    # Generate LaTeX table
    print("\n" + "=" * 120)
    print("LATEX TABLE - Combined Metrics (Models as rows)")
    print("=" * 120)
    latex_str = generate_combined_metrics_latex(combined_precision, combined_recall, combined_f1, datasets, models)
    print(latex_str)
    
    return combined_df

def generate_combined_metrics_latex(combined_precision, combined_recall, combined_f1, datasets, models):
    """
    Generate LaTeX table with models as rows and 3 columns per dataset (precision, recall, F1)
    """
    # Calculate number of columns: Model + 3 columns per dataset
    num_cols = 1 + (3 * len(datasets))
    col_spec = 'l' + 'ccc' * len(datasets)
    
    latex = "\\begin{table}[h!]\n"
    latex += "\\centering\n"
    latex += "\\caption{Combined Precision, Recall, and F1 Score Results by Model}\n"
    latex += "\\label{tab:combined_metrics_models}\n"
    latex += f"\\begin{{tabular}}{{{col_spec}}}\n"
    latex += "\\hline\n"
    
    # Header row with multicolumn for each dataset
    header1 = "Model"
    for dataset in datasets:
        header1 += f" & \\multicolumn{{3}}{{c}}{{{dataset.upper()}}}"
    header1 += " \\\\\n"
    latex += header1
    
    # Sub-header row with Precision, Recall, F1 for each dataset
    header2 = ""
    for dataset in datasets:
        header2 += " & Prec (\\%) & Rec (\\%) & F1 (\\%)"
    header2 += " \\\\\n"
    latex += header2
    latex += "\\hline\n"
    
    # Sort models by the order in MODEL_LATEX_MAPPING
    model_order = list(MODEL_LATEX_MAPPING.keys())
    sorted_models = sorted(models, 
                          key=lambda x: model_order.index(x) if x in model_order else len(model_order))
    
    # Data rows: one per model
    for model in sorted_models:
        # Map model name to LaTeX command
        model_latex = MODEL_LATEX_MAPPING.get(model, model)
        row_str = f"{model_latex}"
        
        for dataset in datasets:
            # Get columns for this dataset
            prec_cols = [col for col in combined_precision.columns if col.startswith(dataset)]
            rec_cols = [col for col in combined_recall.columns if col.startswith(dataset)]
            f1_cols = [col for col in combined_f1.columns if col.startswith(dataset)]
            
            # Precision value
            if prec_cols:
                prec_val = combined_precision.loc[model, prec_cols[0]]
                if prec_val > 0:
                    row_str += f" &  \gradientd{{ {prec_val:.1f} }}"
                else:
                    row_str += " & --"
            else:
                row_str += " & --"
            
            # Recall value
            if rec_cols:
                rec_val = combined_recall.loc[model, rec_cols[0]]
                if rec_val > 0:
                    row_str += f" & \gradientd{{  {rec_val:.1f} }}"
                else:
                    row_str += " & --"
            else:
                row_str += " & --"
            
            # F1 value
            if f1_cols:
                f1_val = combined_f1.loc[model, f1_cols[0]]
                if f1_val > 0:
                    row_str += f" & \gradientd{{ {f1_val:.1f} }}"
                else:
                    row_str += " & --"
            else:
                row_str += " & --"
        
        row_str += " \\\\\n"
        latex += row_str
    
    latex += "\\hline\n"
    latex += "\\end{tabular}\n"
    latex += "\\end{table}\n"
    
    return latex

# Create the combined metrics table
combined_metrics_df = create_combined_metrics_table(combined_precision, combined_recall, combined_f1)


In [ ]:
def create_combined_scene_metrics_table(results):
    """
    Create combined tables for cycle-level precision, recall, and F1 score
    """
    if not results:
        print("No results to create tables from!")
        return None, None, None
    
    # Collect data for each metric by dataset
    precision_data = {}
    recall_data = {}
    f1_data = {}
    
    for dataset_name, df in results.items():
        # Check if required columns exist
        required_cols = ['cycle_precision', 'cycle_recall', 'cycle_f1_score']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f"Missing columns for {dataset_name}: {missing_cols}")
            continue
        
        # Clean model names
        df_clean = df.copy()
        df_clean['model_clean'] = df_clean['model'].apply(lambda x: x.split('/')[-1])
        
        # Group by model and get mean values
        for model in df_clean['model_clean'].unique():
            model_data = df_clean[df_clean['model_clean'] == model]
            
            if model not in precision_data:
                precision_data[model] = {}
                recall_data[model] = {}
                f1_data[model] = {}
            
            # Values are already converted to percentage in load_all_metrics, so just take mean
            precision_data[model][dataset_name] = model_data['cycle_precision'].mean()
            recall_data[model][dataset_name] = model_data['cycle_recall'].mean()
            f1_data[model][dataset_name] = model_data['cycle_f1_score'].mean()
    
    # Create DataFrames from the collected data
    combined_precision = pd.DataFrame(precision_data).T.round(1) if precision_data else None
    combined_recall = pd.DataFrame(recall_data).T.round(1) if recall_data else None
    combined_f1 = pd.DataFrame(f1_data).T.round(1) if f1_data else None
    
    return combined_precision, combined_recall, combined_f1


def create_scene_metrics_display_table(combined_precision, combined_recall, combined_f1):
    """
    Create a display table with models as rows and columns for cycle precision, recall, and F1 per dataset
    """
    if combined_precision is None or combined_recall is None or combined_f1 is None:
        print("No data to create table!")
        return None
    
    print("\nCYCLE-LEVEL METRICS TABLE - Precision, Recall & F1 Score")
    print("=" * 120)
    
    # Get datasets from column names
    datasets = []
    for col in combined_precision.columns:
        dataset = col.split('_')[0]
        if dataset not in datasets:
            datasets.append(dataset)
    
    # Get models
    models = combined_precision.index.tolist()
    
    # Create rows: one per model with 3 columns per dataset
    table_rows = []
    
    for model in models:
        row = {'Model': model}
        
        for dataset in datasets:
            # Get columns for this dataset
            prec_cols = [col for col in combined_precision.columns if col.startswith(dataset)]
            rec_cols = [col for col in combined_recall.columns if col.startswith(dataset)]
            f1_cols = [col for col in combined_f1.columns if col.startswith(dataset)]
            
            if prec_cols:
                prec_val = combined_precision.loc[model, prec_cols[0]]
                row[f"{dataset.upper()}\nPrecision (%)"] = f"{prec_val:.1f}" if prec_val > 0 else "--"
            else:
                row[f"{dataset.upper()}\nPrecision (%)"] = "--"
            
            if rec_cols:
                rec_val = combined_recall.loc[model, rec_cols[0]]
                row[f"{dataset.upper()}\nRecall (%)"] = f"{rec_val:.1f}" if rec_val > 0 else "--"
            else:
                row[f"{dataset.upper()}\nRecall (%)"] = "--"
            
            if f1_cols:
                f1_val = combined_f1.loc[model, f1_cols[0]]
                row[f"{dataset.upper()}\nF1 Score (%)"] = f"{f1_val:.1f}" if f1_val > 0 else "--"
            else:
                row[f"{dataset.upper()}\nF1 Score (%)"] = "--"
        
        table_rows.append(row)
    
    # Create DataFrame
    combined_df = pd.DataFrame(table_rows)
    
    # Print the table
    print("\nTable Format: Models (rows) × Dataset Metrics (columns)")
    print("-" * 120)
    print(combined_df.to_string(index=False))
    
    # Generate LaTeX table
    print("\n" + "=" * 120)
    print("LATEX TABLE - Cycle-Level Metrics (Models as rows)")
    print("=" * 120)
    latex_str = generate_combined_metrics_latex(combined_precision, combined_recall, combined_f1, datasets, models)
    print(latex_str)
    
    return combined_df


# Create cycle-level metrics tables
cycle_precision, cycle_recall, cycle_f1 = create_combined_scene_metrics_table(results)
cycle_metrics_df = create_scene_metrics_display_table(cycle_precision, cycle_recall, cycle_f1)

### Cycle-Level Metrics Table